## Load all packages and functions

In [1]:
import cv2 # to install on mac: pip install opencv-python
from scipy.interpolate import interp1d # for interpolating points
from sklearn.decomposition import PCA # for principal component analysis
from scipy.spatial import procrustes # for Procrustes analysis
from scipy.spatial import ConvexHull # for convex hull
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # for LDA
from sklearn.metrics import confusion_matrix # for confusion matrix
import os
from os import listdir # for retrieving files from directory
from os.path import isfile, join # for retrieving files from directory
import matplotlib.pyplot as plt # for plotting
%matplotlib inline
import numpy as np # for using arrays
import math # for mathematical operations
import pandas as pd # for using pandas dataframes
import seaborn as sns # for plotting in seaborna
import csv
from statistics import mean, stdev
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold
from sklearn import linear_model
from sklearn import datasets

In [2]:
def angle_between(p1, p2, p3):
    """
    define a function to find the angle between 3 points anti-clockwise in degrees, p2 being the vertex
    inputs: three angle points, as tuples
    output: angle in degrees
    """
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    deg1 = (360 + math.degrees(math.atan2(x1 - x2, y1 - y2))) % 360
    deg2 = (360 + math.degrees(math.atan2(x3 - x2, y3 - y2))) % 360
    return deg2 - deg1 if deg1 <= deg2 else 360 - (deg1 - deg2)

def rotate_points(xvals, yvals, degrees):
    """"
    define a function to rotate 2D x and y coordinate points around the origin
    inputs: x and y vals (can take pandas dataframe columns) and the degrees (positive, anticlockwise) to rotate
    outputs: rotated and y vals
    """
    angle_to_move = 90-degrees
    rads = np.deg2rad(angle_to_move)
    
    new_xvals = xvals*np.cos(rads)-yvals*np.sin(rads)
    new_yvals = xvals*np.sin(rads)+yvals*np.cos(rads)
    
    return new_xvals, new_yvals

def interpolation(x, y, number): 
    """
    define a function to return equally spaced, interpolated points for a given polyline
    inputs: arrays of x and y values for a polyline, number of points to interpolate
    ouputs: interpolated points along the polyline, inclusive of start and end points
    """
    distance = np.cumsum(np.sqrt( np.ediff1d(x, to_begin=0)**2 + np.ediff1d(y, to_begin=0)**2 ))
    distance = distance/distance[-1]

    fx, fy = interp1d( distance, x ), interp1d( distance, y )

    alpha = np.linspace(0, 1, number)
    x_regular, y_regular = fx(alpha), fy(alpha)
    
    return x_regular, y_regular

def euclid_dist(x1, y1, x2, y2):
    """
    define a function to return the euclidean distance between two points
    inputs: x and y values of the two points
    output: the eulidean distance
    """
    return np.sqrt((x2-x1)**2 + (y2-y1)**2)

def poly_area(x,y):
    """
    define a function to calculate the area of a polygon using the shoelace algorithm
    inputs: separate numpy arrays of x and y coordinate values
    outputs: the area of the polygon
    """
    return 0.5*np.abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))

def gpa_mean(leaf_arr, landmark_num, dim_num):
    
    """
    define a function that given an array of landmark data returns the Generalized Procrustes Analysis mean
    inputs: a 3 dimensional array of samples by landmarks by coordinate values, number of landmarks, number of dimensions
    output: an array of the Generalized Procrustes Analysis mean shape
    
    """

    ref_ind = 0 # select arbitrary reference index to calculate procrustes distances to
    ref_shape = leaf_arr[ref_ind, :, :] # select the reference shape

    mean_diff = 10**(-30) # set a distance between means to stop the algorithm

    old_mean = ref_shape # for the first comparison between means, set old_mean to an arbitrary reference shape

    d = 1000000 # set d initially arbitraily high

    while d > mean_diff: # set boolean criterion for Procrustes distance between mean to stop calculations

        arr = np.zeros( ((len(leaf_arr)),landmark_num,dim_num) ) # empty 3D array: # samples, landmarks, coord vals

        for i in range(len(leaf_arr)): # for each leaf shape 

            s1, s2, distance = procrustes(old_mean, leaf_arr[i]) # calculate procrustes adjusted shape to ref for current leaf
            arr[i] = s2 # store procrustes adjusted shape to array

        new_mean = np.mean(arr, axis=(0)) # calculate mean of all shapes adjusted to reference

        s1, s2, d = procrustes(old_mean, new_mean) # calculate procrustes distance of new mean to old mean

        old_mean = new_mean # set the old_mean to the new_mea before beginning another iteration

    return new_mean

def Circularity(shape_arr):
    
    lines = np.hstack([shape_arr,np.roll(shape_arr,-1,axis=0)])
    area = 0.5*abs(sum(x1*y2-x2*y1 for x1,y1,x2,y2 in lines))
    
    distance = np.cumsum(np.sqrt( np.ediff1d(shape_arr[:,0], to_begin=0)**2 + np.ediff1d(shape_arr[:,1], to_begin=0)**2 ))
    perimeter = distance[-1]
    
    circularity = (4*math.pi*area)/perimeter**2
    
    return circularity


In [3]:
def process_leaf_data(mdata):
    ######################
    ### SET PARAMETERS ###
    ######################

    high_res_pts = 1000
    res = 50  # Total pseudo-landmarks = 2 * res - 1

    # Create empty array to store all leaves' cm-scaled pseudo-landmarks
    cm_arr = np.zeros((len(mdata), (res * 2) - 1, 2))

    # Process each leaf
    for lf in range(len(mdata)):

        ###############################
        ### READ IN GRAYSCALE IMAGE ###
        ###############################

        curr_image = mdata["file"][lf]
        print(lf, curr_image)

        img = cv2.bitwise_not(cv2.cvtColor(cv2.imread(data_dir + curr_image),cv2.COLOR_BGR2GRAY))
        contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        ##############################
        ### SELECT LARGEST CONTOUR ###
        ##############################

        x_conts, y_conts, areas_conts = [], [], []

        for c in contours:
            x_vals = [pt[0][0] for pt in c]
            y_vals = [pt[0][1] for pt in c]
            area = (max(x_vals) - min(x_vals)) * (max(y_vals) - min(y_vals))
            x_conts.append(x_vals)
            y_conts.append(y_vals)
            areas_conts.append(area)

        area_inds = np.flip(np.argsort(areas_conts))
        sorted_x_conts = np.array(x_conts, dtype=object)[area_inds]
        sorted_y_conts = np.array(y_conts, dtype=object)[area_inds]

        ################################################
        ### INTERPOLATE HIGH RES NUMBER OF LANDMARKS ###
        ################################################

        high_res_x, high_res_y = interpolation(
            sorted_x_conts[0], sorted_y_conts[0], high_res_pts
        )

        ###############################
        ### FIND BASE AND TIP INDEX ###
        ###############################

        base_pt = np.array((mdata["base_x"][lf], mdata["base_y"][lf]))
        tip_pt = np.array((mdata["tip_x"][lf], mdata["tip_y"][lf]))

        base_dists = [
            euclid_dist(base_pt[0], base_pt[1], high_res_x[pt], high_res_y[pt])
            for pt in range(len(high_res_x))
        ]
        tip_dists = [
            euclid_dist(tip_pt[0], tip_pt[1], high_res_x[pt], high_res_y[pt])
            for pt in range(len(high_res_x))
        ]

        base_ind = np.argmin(base_dists)
        tip_ind = np.argmin(tip_dists)

        ################################
        ### RESET BASE INDEX TO ZERO ###
        ################################

        high_res_x = np.concatenate((high_res_x[base_ind:], high_res_x[:base_ind]))
        high_res_y = np.concatenate((high_res_y[base_ind:], high_res_y[:base_ind]))

        tip_ind = tip_ind - base_ind
        base_ind = 0

        lf_contour = np.column_stack((high_res_x, high_res_y))

        ##############################################################
        ### INTERPOLATE EACH SIDE WITH DESIRED NUMBER OF LANDMARKS ###
        ##############################################################

        left_inter_x, left_inter_y = interpolation(
            lf_contour[base_ind:tip_ind+1, 0], lf_contour[base_ind:tip_ind+1, 1], res
        )
        right_inter_x, right_inter_y = interpolation(
            lf_contour[tip_ind:, 0], lf_contour[tip_ind:, 1], res
        )

        left_inter_x = np.delete(left_inter_x, -1)
        left_inter_y = np.delete(left_inter_y, -1)

        lf_pts_left = np.column_stack((left_inter_x, left_inter_y))
        lf_pts_right = np.column_stack((right_inter_x, right_inter_y))
        lf_pts = np.row_stack((lf_pts_left, lf_pts_right))

        ##########################################################
        ### ROTATE LEAVES UPWARD AND SCALE SIZE TO CENTIMETERS ###
        ##########################################################

        tip_point = lf_pts[res-1, :]
        base_point = lf_pts[0, :]

        ang = angle_between(tip_point, base_point, (base_point[0]+1, base_point[1]))

        rot_x, rot_y = rotate_points(lf_pts[:, 0], lf_pts[:, 1], ang)
        rot_pts = np.column_stack((rot_x, rot_y))

        cm_lf = rot_pts / mdata["px_cm"][lf]
        cm_arr[lf, :, :] = cm_lf

    return cm_arr

## Load in data

In [4]:
#change working directory 
os.chdir('/Volumes/external/temp_drought_all_062325/data_020206/')

### If reloading data: 

In [5]:
cm_arr = np.load('cm_arr.npy')
proc_arr = np.load('proc_arr.npy')
flat_arr = np.load('flat_arr.npy')

### First time run: 

In [6]:
mdata = pd.read_csv("mdata_020226.csv") # read in csv

mdata.head() # head data to check

,file,dataset,condition,px_cm,genotype,population,location,node,rep_number,base_x,base_y,tip_x,tip_y,rosette_id
0,11.154_15_16_rep3_20240529_0001_lf1.jpg,temp,16,39.370079,NY_65,11,NYC,1,3.0,476.0,128.0,37.5,122.5,aa
1,11.154_15_16_rep3_20240529_0001_lf2.jpg,temp,16,39.370079,NY_65,11,NYC,2,3.0,473.0,177.0,33.5,165.5,aa
2,11.154_15_16_rep3_20240529_0001_lf3.jpg,temp,16,39.370079,NY_65,11,NYC,3,3.0,481.0,107.0,38.5,108.5,aa
3,11.154_15_16_rep3_20240529_0001_lf4.jpg,temp,16,39.370079,NY_65,11,NYC,4,3.0,475.0,116.0,38.5,117.5,aa
4,11.154_15_16_rep3_20240529_0001_lf5.jpg,temp,16,39.370079,NY_65,11,NYC,5,3.0,482.0,123.0,32.5,131.5,aa


In [7]:
data_dir = "./all_jpgs/" # set data directory

file_names = [f for f in listdir(data_dir) if isfile(join(data_dir, f))] # create a list of file names

#file_names.remove('.DS_Store') # remove .DS_Store file

file_names.sort() # sort the list of file names

file_names # check list of file names

['11.154_15_16_rep3_20240529_0001_lf1.jpg',
 '11.154_15_16_rep3_20240529_0001_lf10.jpg',
 '11.154_15_16_rep3_20240529_0001_lf11.jpg',
 '11.154_15_16_rep3_20240529_0001_lf2.jpg',
 '11.154_15_16_rep3_20240529_0001_lf3.jpg',
 '11.154_15_16_rep3_20240529_0001_lf4.jpg',
 '11.154_15_16_rep3_20240529_0001_lf5.jpg',
 '11.154_15_16_rep3_20240529_0001_lf6.jpg',
 '11.154_15_16_rep3_20240529_0001_lf7.jpg',
 '11.154_15_16_rep3_20240529_0001_lf8.jpg',
 '11.154_15_16_rep3_20240529_0001_lf9.jpg',
 '11.154_15_16_rep5_20240529_0001_lf1.jpg',
 '11.154_15_16_rep5_20240529_0001_lf10.jpg',
 '11.154_15_16_rep5_20240529_0001_lf2.jpg',
 '11.154_15_16_rep5_20240529_0001_lf3.jpg',
 '11.154_15_16_rep5_20240529_0001_lf4.jpg',
 '11.154_15_16_rep5_20240529_0001_lf5.jpg',
 '11.154_15_16_rep5_20240529_0001_lf6.jpg',
 '11.154_15_16_rep5_20240529_0001_lf7.jpg',
 '11.154_15_16_rep5_20240529_0001_lf8.jpg',
 '11.154_15_16_rep5_20240529_0001_lf9.jpg',
 '11.154_15_16_rep6_20240520_0001_lf1.jpg',
 '11.154_15_16_rep6_20240520_

#### Rotate, scale, and add pseudo-landmarks

In [ ]:
# the number of equidistant points to create
# an initial high resolution outline of the leaf
high_res_pts = 1000 

# the ultimate number of equidistant points on each side of the leaf
# (-1 for the tip)
# the leaf will have res*2-1 pseudo-landmarks
#################
#################
#################
res = 50 ########
#################
#################
#################

# an array to store pseudo-landmarks
cm_arr = np.zeros((len(mdata),(res*2)-1,2))

# for each leaf . . .
for lf in range(len(mdata)):

    ###############################
    ### READ IN GRAYSCALE IMAGE ###
    ###############################

    curr_image = mdata["file"][lf] # select the current image
    print(lf, curr_image) # print each leaf in case there are problems later

    # read in image
    # convert to grayscale
    # invert the binary
    img = cv2.bitwise_not(cv2.cvtColor(cv2.imread(data_dir + curr_image),cv2.COLOR_BGR2GRAY))

    # find contours of binary objects
    contours, hierarchy = cv2.findContours(img,  
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) 

    ##############################
    ### SELECT LARGEST CONTOUR ###
    ##############################

    # ideally there is only one leaf in the image
    # in the case there are smaller objects
    # this code selects the largest object (the leaf)
    # if there is one and only one object in the image
    # then the following code is not necessary

    x_conts = [] # list of lists of contour x vals
    y_conts = [] # list of lists of contour y vals
    areas_conts = [] # list of bounding box areas of contours
    for c in contours: # for each contour
        x_vals = [] # store x vals for current contour 
        y_vals = [] # store y vals for current contour
        for i in range(len(c)): # for each point in current contour
            x_vals.append(c[i][0][0]) # isolate x val
            y_vals.append(c[i][0][1]) # isolate y val
        area = (max(x_vals) - min(x_vals))*(max(y_vals) - min(y_vals)) # calculate bounding box area of contour
        x_conts.append(x_vals) # append the current contour x vals
        y_conts.append(y_vals) # append the current contour y vals
        areas_conts.append(area) # append the current contour bounding box areas

    area_inds = np.flip(np.argsort(areas_conts)) # get indices to sort contours by area
    sorted_x_conts = np.array(x_conts, dtype=object)[area_inds][0:] # areas sorted largest to smallest, x vals
    sorted_y_conts = np.array(y_conts, dtype=object)[area_inds][0:] # areas sorted largest to smallest, y vals

    ################################################
    ### INTERPOLATE HIGH RES NUMBER OF LANDMARKS ###
    ################################################

    # convert the leaf to high resolution number of landmarks
    # using high_res_pt value
    # need to convert arrays of pixel int to floats first
    high_res_x, high_res_y = interpolation(sorted_x_conts[0], 
                                           sorted_y_conts[0], high_res_pts)

    ###############################
    ### FIND BASE AND TIP INDEX ###
    ###############################

    # get the base and tip landmark point values
    base_pt = np.array((mdata["base_x"][lf], mdata["base_y"][lf]))
    tip_pt = np.array((mdata["tip_x"][lf], mdata["tip_y"][lf]))

    base_dists = [] # store distance of each high res point to base
    tip_dists = [] # store distance of each high res point to tip

    for pt in range(len(high_res_x)): # for each of the high resolution points

        # euclidean distance of the current point from the base and tip landmark
        ed_base = euclid_dist(base_pt[0], base_pt[1], high_res_x[pt], high_res_y[pt])
        ed_tip = euclid_dist(tip_pt[0], tip_pt[1], high_res_x[pt], high_res_y[pt])

        # store distance of current point from base/tip
        base_dists.append(ed_base)
        tip_dists.append(ed_tip)

    # get index of base and tip points
    base_ind = np.argmin(base_dists)
    tip_ind = np.argmin(tip_dists)

    ################################
    ### RESET BASE INDEX TO ZERO ###
    ################################

    # reset base index position to zero
    high_res_x = np.concatenate((high_res_x[base_ind:],high_res_x[:base_ind]))
    high_res_y = np.concatenate((high_res_y[base_ind:],high_res_y[:base_ind]))

    # recalculate indices with new indexing
    tip_ind = tip_ind-base_ind # note: negative index if tip_ind<base_ind
    base_ind = base_ind-base_ind

    # create single array for leaf coordinates
    lf_contour = np.column_stack((high_res_x, high_res_y))

    ##############################################################
    ### INTERPOLATE EACH SIDE WITH DESIRED NUMBER OF LANDMARKS ###
    ##############################################################

    # interpolate at desired resolution the left and right sides of the leaf
    left_inter_x, left_inter_y = interpolation(lf_contour[base_ind:tip_ind+1,0],lf_contour[base_ind:tip_ind+1,1],res)
    right_inter_x, right_inter_y = interpolation(lf_contour[tip_ind:,0],lf_contour[tip_ind:,1],res)

    # the start of the right side and end of the left side
    # both contain the tip landmark
    # delete the last point on the left side
    left_inter_x = np.delete(left_inter_x, -1)
    left_inter_y = np.delete(left_inter_y, -1)

    # BASE OF LEAF IS INDEX 0
    # TIP INDEX IS RES-1 IF BOTH LEFT & RIGHT POINTS
    # TOTAL PSEUDOLANDMARKS IS 2*RES-1
    lf_pts_left = np.column_stack((left_inter_x, left_inter_y))
    lf_pts_right = np.column_stack((right_inter_x, right_inter_y))
    lf_pts = np.row_stack((lf_pts_left, lf_pts_right))

    ##########################################################
    ### ROTATE LEAVES UPWARD AND SCALE SIZE TO CENTIMETERS ###
    ##########################################################

    tip_point = lf_pts[res-1,:] # get tip point
    base_point = lf_pts[0,:] # get base point

    # calculate angle between tip. base, and an arbitrary reference
    ang = angle_between(tip_point, base_point, (base_point[0]+1,base_point[1]) )

    # rotate points upwards
    rot_x, rot_y = rotate_points(lf_pts[:,0], lf_pts[:,1], ang) 
    rot_pts = np.column_stack((rot_x, rot_y))

    # scale leaf into cm
    cm_lf = rot_pts/(mdata["px_cm"][lf])
    
    # store the leaf scaled into cm into the cm_arr
    cm_arr[lf,:,:] = cm_lf

#### Procrustes distance and PCA

In [8]:
res = 50
landmark_num = (res*2)-1 # select number of landmarks
dim_num = 2 # select number of coordinate value dimensions

##########################
### CALCULATE GPA MEAN ###
##########################

mean_shape = gpa_mean(cm_arr, landmark_num, dim_num)

################################
### ALIGN LEAVES TO GPA MEAN ###
################################

# array to store Procrustes aligned shapes
proc_arr = np.zeros(np.shape(cm_arr)) 

for i in range(len(cm_arr)):
    s1, s2, distance = procrustes(mean_shape, cm_arr[i,:,:]) # calculate procrustes adjusted shape to ref for current leaf
    proc_arr[i] = s2 # store procrustes adjusted shape to array
    
#################################################
### FIRST, CALCULATE PERCENT VARIANCE ALL PCs ###
#################################################

######
PC_NUMBER = np.shape(proc_arr)[1] # PC number = number of leaves
#######

# use the reshape function to flatten to 2D
flat_arr = proc_arr.reshape(np.shape(proc_arr)[0], 
                                 np.shape(proc_arr)[1]*np.shape(proc_arr)[2]) 

pca_all = PCA(n_components=2) 
PCs_all = pca_all.fit_transform(flat_arr) # fit a PCA for all data

# print out explained variance for each PC
print("PC: " + "var, " + "overall ") 
for i in range(len(pca_all.explained_variance_ratio_)):
    print("PC" + str(i+1) + ": " + str(round(pca_all.explained_variance_ratio_[i]*100,1)) + 
          "%, " + str(round(pca_all.explained_variance_ratio_.cumsum()[i]*100,1)) + "%"  )

#################################################
### NEXT, CALCULATE THE DESIRED NUMBER OF PCs ###
#################################################

######
PC_NUMBER = 2 # PC number = 2, for limiting to 2 axes for morphospace reconstruction
#######

pca = PCA(n_components=PC_NUMBER) 
PCs = pca.fit_transform(flat_arr) # fit a PCA for only desired PCs

# print out explained variance for each PC
print("PC: " + "var, " + "overall ") 
for i in range(len(pca.explained_variance_ratio_)):
    print("PC" + str(i+1) + ": " + str(round(pca.explained_variance_ratio_[i]*100,1)) + 
          "%, " + str(round(pca.explained_variance_ratio_.cumsum()[i]*100,1)) + "%"  )
    
# add PCs to dataframe for plotting
mdata["PC1"] = PCs[:,0]
mdata["PC2"] = PCs[:,1]

PC: var, overall 
PC1: 24.0%, 24.0%
PC2: 19.0%, 43.0%
PC: var, overall 
PC1: 24.0%, 24.0%
PC2: 19.0%, 43.0%


#### Save array files

In [ ]:
# Save rotated leaves with pseudo-landmarks
np.save('cm_arr', cm_arr)
#save procurestes aligned leaves
np.save('proc_arr', proc_arr)
#save flattened procurestes aligned leaves
np.save('flat_arr', flat_arr)

#### Save Pseudo-landmarks in spreadsheet

In [ ]:
np.shape(proc_arr)

In [ ]:
dataset = mdata['dataset']
condition = mdata['condition']
node = mdata['node']
genotype = mdata['genotype']

In [ ]:
# reshape pseuo-landmark array to long format
proc_arrdf = np.reshape(proc_arr, (5018, 99*2))

# check formatting 
print(np.shape(proc_arrdf))

# save new array with file names and check
df4 = np.column_stack([file_names, dataset, condition, node, genotype, proc_arrdf])
print(np.shape(df4))

# save final array as dataframe 
df_out_proc = pd.DataFrame(df4)
df_out_proc.to_csv('/Volumes/external/temp_drought_all_062325/data_020206/proc_df.csv')

# Generate Mean Leaf by temperature condition and genotype within temperature condition

In [ ]:
temp30_arr = process_leaf_data(temp30)
mean_temp30 = gpa_mean(temp30_arr, landmark_num, dim_num)

In [ ]:
np.shape(mean_t30n1)

In [ ]:
# reshape pseuo-landmark array to long format
test1 = np.reshape(mean_t30n16, (99*2))

# check formatting 
print(np.shape(test1))

# save new array with file names and check
df4 = np.column_stack([test1])
print(np.shape(df4))

# save final array as dataframe 
df_out_proc = pd.DataFrame(df4)
df_out_proc.to_csv('/Volumes/external/temp_drought_all_062325/data_020206/mean_t30n16_proc_df.csv')

# Find Euclidean distance between mean leaves

In [ ]:
df1 = pd.read_csv('proc_long.csv')
df2 = pd.read_csv('temp16_long.csv')
print(len(df1))
print(df1.head())
print(len(df2))
print(df2.head())

In [ ]:
df2["shape_id"] = df2.index // 99

In [ ]:
df1 = df1.sort_values("point").reset_index(drop=True)
df2 = df2.sort_values(["shape_id", "point"]).reset_index(drop=True)

In [ ]:
merged = df2.merge(df1, on="point", suffixes=("_df2", "_df1"))

In [ ]:
merged.head()

In [ ]:
merged["distance"] = np.sqrt(
    (merged["x_df2"] - merged["x_df1"])**2 +
    (merged["y_df2"] - merged["y_df1"])**2
)

In [ ]:
distance_df = merged[["shape_id_df1", "point", "distance", "dataset", "condition", "node", "genotype", "x_df1", "y_df1"]]

In [ ]:
distance_temp = distance_df[(distance_df['dataset'] == "temp")]
print(distance_temp.head())
print(len(distance_temp))

In [ ]:
plt.fill(mean_temp30[:,0], mean_temp30[:,1], c = "lightgray")
plt.gca().set_aspect("equal")
plt.axis("off")

sns.scatterplot(data=merg_2030_df, x="x_df1", y="y_df1", hue= "distance")
plt.legend(bbox_to_anchor=(1.00, 1.02), prop={'size': 8.9})

plt.savefig('meanleaf30_pointstemp20_021826.png', dpi= 300, bbox_inches='tight')

# Eculidean distance by development 

In [ ]:
df6 = pd.read_csv('t16n1n_long.csv')
df7 = pd.read_csv('t16n4n_long.csv')
df8 = pd.read_csv('t16n8n_long.csv')
df9 = pd.read_csv('t16n12n_long.csv')
df10 = pd.read_csv('t16n16n_long.csv')

#df6 = pd.read_csv('t30n1n_long.csv')
#df7 = pd.read_csv('t30n4n_long.csv')
#df8 = pd.read_csv('t30n8n_long.csv')
#df9 = pd.read_csv('t30n12n_long.csv')
#df10 = pd.read_csv('t30n16n_long.csv')

In [ ]:
#df1 = df1.sort_values("point").reset_index(drop=True)
#df2 = df2.sort_values("point").reset_index(drop=True)
#df3 = df3.sort_values("point").reset_index(drop=True)
#df4 = df4.sort_values("point").reset_index(drop=True)
#df5 = df5.sort_values("point").reset_index(drop=True)
df6 = df6.sort_values("point").reset_index(drop=True)
df7 = df7.sort_values("point").reset_index(drop=True)
df8 = df8.sort_values("point").reset_index(drop=True)
df9 = df9.sort_values("point").reset_index(drop=True)
df10 = df10.sort_values("point").reset_index(drop=True)

In [ ]:
merged_1 = df7.merge(df6, on="point", suffixes=("_df7", "_df6"))
merged_2 = df8.merge(df6, on="point", suffixes=("_df8", "_df6"))
merged_3 = df9.merge(df6, on="point", suffixes=("_df9", "_df6"))
merged_4 = df10.merge(df6, on="point", suffixes=("_df10", "_df6"))
merged_5 = df8.merge(df7, on="point", suffixes=("_df8", "_df7"))
merged_6 = df9.merge(df7, on="point", suffixes=("_df9", "_df7"))
merged_7 = df10.merge(df7, on="point", suffixes=("_df10", "_df7"))
merged_8 = df9.merge(df8, on="point", suffixes=("_df9", "_df8"))
merged_9 = df10.merge(df8, on="point", suffixes=("_df10", "_df8"))
merged_10 = df10.merge(df9, on="point", suffixes=("_df10", "_df9"))

In [ ]:
merged_1["distance"] = np.sqrt(
    (merged_1["x_df7"] - merged_1["x_df6"])**2 +
    (merged_1["y_df7"] - merged_1["y_df6"])**2
)

merged_2["distance"] = np.sqrt(
    (merged_2["x_df8"] - merged_2["x_df6"])**2 +
    (merged_2["y_df8"] - merged_2["y_df6"])**2
)

merged_3["distance"] = np.sqrt(
    (merged_3["x_df9"] - merged_3["x_df6"])**2 +
    (merged_3["y_df9"] - merged_3["y_df6"])**2
)

merged_4["distance"] = np.sqrt(
    (merged_4["x_df10"] - merged_4["x_df6"])**2 +
    (merged_4["y_df10"] - merged_4["y_df6"])**2
)

merged_5["distance"] = np.sqrt(
    (merged_5["x_df8"] - merged_5["x_df7"])**2 +
    (merged_5["y_df8"] - merged_5["y_df7"])**2
)

merged_6["distance"] = np.sqrt(
    (merged_6["x_df9"] - merged_6["x_df7"])**2 +
    (merged_6["y_df9"] - merged_6["y_df7"])**2
)

merged_7["distance"] = np.sqrt(
    (merged_7["x_df10"] - merged_7["x_df7"])**2 +
    (merged_7["y_df10"] - merged_7["y_df7"])**2
)

merged_8["distance"] = np.sqrt(
    (merged_8["x_df9"] - merged_8["x_df8"])**2 +
    (merged_8["y_df9"] - merged_8["y_df8"])**2
)

merged_9["distance"] = np.sqrt(
    (merged_9["x_df10"] - merged_9["x_df8"])**2 +
    (merged_9["y_df10"] - merged_9["y_df8"])**2
)

merged_10["distance"] = np.sqrt(
    (merged_10["x_df10"] - merged_10["x_df9"])**2 +
    (merged_10["y_df10"] - merged_10["y_df9"])**2
)

In [ ]:
merged_1_df = merged_1[["point", "distance", "x_df7", "y_df7"]]
merged_2_df = merged_2[["point", "distance", "x_df8", "y_df8"]]
merged_3_df = merged_3[["point", "distance", "x_df9", "y_df9"]]
merged_4_df = merged_4[["point", "distance", "x_df10", "y_df10"]]
merged_5_df = merged_5[["point", "distance", "x_df8", "y_df8"]]
merged_6_df = merged_6[["point", "distance", "x_df9", "y_df9"]]
merged_7_df = merged_7[["point", "distance", "x_df10", "y_df10"]]
merged_8_df = merged_8[["point", "distance", "x_df9", "y_df9"]]
merged_9_df = merged_9[["point", "distance", "x_df10", "y_df10"]]
merged_10_df = merged_10[["point", "distance", "x_df10", "y_df10"]]

In [ ]:
plt.fill(mean_t16n12[:,0], mean_t16n12[:,1], c = "lightgray")
plt.gca().set_aspect("equal")
plt.axis("off")

sns.scatterplot(data=merged_10_df, x="x_df10", y="y_df10", hue= "distance")
plt.legend(bbox_to_anchor=(1.00, 1.02), prop={'size': 8.9})

plt.savefig('mean_t16n12_pointsn16_021826.png', dpi= 300, bbox_inches='tight')